# 09 - Prompt Tuning

Demonstrates `ClassificationPromptTuner` — a hill-climbing loop that iteratively improves a prompt by asking the model to fix its own mistakes.

The tuner:
1. Applies the current prompt to a labeled dataset
2. Evaluates accuracy (precision, recall)
3. If not perfect, sends the wrong examples back to the model and asks for an improved prompt
4. Repeats until 100% accuracy or `max_iterations` is reached

Optionally persists each improvement to a `PromptCatalog` for version tracking.

## A - Setup

Create a model client and build a small labeled dataset. The dataset must have:
- `content` column — the text to classify
- `actual_class` column — boolean ground truth

In [ ]:
import pandas as pd
from aimu.prompts import ClassificationPromptTuner

# Use any ModelClient -- swap in AnthropicClient, OpenAIClient, etc.
# from aimu.models import AnthropicClient, AnthropicModel
# model_client = AnthropicClient(AnthropicModel.CLAUDE_SONNET_4_5)
from aimu.models import OllamaClient, OllamaModel

model_client = OllamaClient(OllamaModel.QWEN_3_8B)

# Labeled dataset: does this text discuss AI / machine learning?
df = pd.DataFrame(
    {
        "content": [
            "Large language models are trained on vast amounts of text data.",
            "The recipe calls for two cups of flour and one egg.",
            "Transformers revolutionised natural language processing in 2017.",
            "She planted tomatoes and basil in her garden this spring.",
            "Reinforcement learning from human feedback improves model alignment.",
            "The train departs from platform 3 at half past nine.",
            "Neural networks learn hierarchical representations from raw data.",
            "He spent the afternoon reading a novel by the lake.",
        ],
        "actual_class": [True, False, True, False, True, False, True, False],
    }
)

print(f"Dataset: {len(df)} rows, {df.actual_class.sum()} positive, {(~df.actual_class).sum()} negative")

## B - Initial Prompt

Write a starting prompt. It must contain a `{content}` placeholder and instruct the model to respond with `[YES]` or `[NO]`.

In [ ]:
initial_prompt = (
    "Does the following text discuss artificial intelligence or machine learning?\n"
    "Answer with [YES] if it does, or [NO] if it does not.\n"
    "Only include [YES] or [NO] in your answer -- nothing else.\n"
    "Text: {content}"
)

tuner = ClassificationPromptTuner(model_client=model_client)

# Preview baseline performance before tuning
baseline_df = tuner.classify_data(initial_prompt, df)
baseline_metrics = tuner.evaluate_results(baseline_df)
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.3f}")

## C - Run the Tuning Loop

`tune()` runs the hill-climbing loop. Each iteration that improves accuracy is logged to stdout.
The loop stops when accuracy reaches 100% or `max_iterations` is exhausted.

In [ ]:
best_prompt = tuner.tune(
    training_data=df,
    initial_prompt=initial_prompt,
    max_iterations=10,
    max_examples=4,
)

print("\n--- Best prompt found ---")
print(best_prompt)

## D - Evaluate the Best Prompt

Run the best prompt on the full dataset and compare with the baseline.

In [ ]:
final_df = tuner.classify_data(best_prompt, df)
final_metrics = tuner.evaluate_results(final_df)

print("Baseline vs. tuned:")
for k in ["accuracy", "precision", "recall"]:
    b = baseline_metrics[k]
    f = final_metrics[k]
    delta = f - b
    sign = "+" if delta >= 0 else ""
    print(f"  {k:12s}  baseline={b:.3f}  tuned={f:.3f}  ({sign}{delta:.3f})")

## E - Catalog Persistence (Optional)

`PromptCatalog` stores each improved prompt version in a SQLite database so you can compare
versions, roll back, and track metrics over time. Pass `catalog` and `prompt_name` to `tune()`
to enable automatic saving.

In [ ]:
import tempfile, os
from aimu.prompts import PromptCatalog

db = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
db.close()

catalog = PromptCatalog(db.name)
prompt_name = "ai-classifier"

best_with_catalog = tuner.tune(
    training_data=df,
    initial_prompt=initial_prompt,
    max_iterations=10,
    catalog=catalog,
    prompt_name=prompt_name,
)

# Show all saved versions
model_id = str(model_client.model.value)
versions = catalog.retrieve_all(prompt_name, model_id)
print(f"Saved {len(versions)} prompt version(s) to catalog:")
for v in versions:
    acc = v.metrics.get("accuracy", "n/a") if v.metrics else "n/a"
    print(f"  v{v.version}  accuracy={acc}")
    print(f"  {v.prompt[:80]}..." if len(v.prompt) > 80 else f"  {v.prompt}")
    print()

os.unlink(db.name)